In [1]:
import seaborn as sns
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

### Функция кодирования категориальных признаков

In [2]:
def categorical_encoding(categorical, df, target, threshold):
    for col in categorical:
        if df[col].nunique() < threshold:
            one_hot = pd.get_dummies(
                df[col], prefix=col, drop_first=True, dtype=int)
            df = pd.concat((df.drop(col, axis=1), one_hot), axis=1)
            df = df.drop(col, axis=1)
        else:
            mean_target = df.groupby(col)[target].mean()
            df[col] = df[col].map(mean_target)
    return df

### Чтение данных, переименование колонок

In [3]:
df = pd.read_csv('ks.csv')

In [ ]:
df.head()

In [5]:
columns = df.columns

In [ ]:
columns

In [7]:
df = df.rename(columns={'Название': 'Title', 'Категория': 'Category', 'Главная категория': 'Main_category', 'Валюта': 'Currency', 'Дедлайн': 'Due_date', 'Дата публикации': 'Pub_date',
                        'Состояние': 'Destiny', 'Инвесторов': 'Investors', 'Страна': 'Country', 'Собрано в долларах': 'Money_collected', 'Цель в долларах': 'Money_targeted'})

In [ ]:
df.head()

### Выделение и кодирование таргета

In [ ]:
df['Destiny'].value_counts()

In [10]:
df.loc[(df['Destiny'] == 'live'), 'target'] = 1

In [11]:
df.loc[(df['Destiny'] == 'suspended'), 'target'] = 0

In [ ]:
df['target'].value_counts()

### Выделение признаков из дат

In [13]:
df['Due_date'] = pd.to_datetime(df['Due_date'])
df['Pub_date'] = pd.to_datetime(df['Pub_date'])
df['time_to_collect'] = (df['Due_date'] - df['Pub_date']).dt.days
df['day_of_pub'] = df['Pub_date'].dt.day_of_week

In [ ]:
df.head()

In [ ]:
df['target'] = df['target'].astype('int16')

In [124]:
df = df.drop(['Destiny', 'Pub_date', 'Due_date'], axis=1)

In [ ]:
df['Main_category'].value_counts()

In [ ]:
df['Category'].value_counts().head(5)

### Кодирование цифровых признаков

In [42]:
numeric = df.describe().columns

In [ ]:
sns.heatmap(df[numeric].corr(method='kendall'), annot=True, fmt='.2f')

In [35]:
import scipy.stats as st

In [ ]:
st.normaltest(df['Money_collected'])

In [ ]:
sns.distplot(df['Investors'])

In [125]:
df = df.drop('Investors', axis=1)

### Кодирование категориальных признаков

In [126]:
categorical = df.describe(include='object').columns

In [ ]:
df.describe(include='object')

In [47]:
cat = categorical.to_list()

In [ ]:
type(cat)

In [128]:
for col in ['Main_category', 'Currency']:
    one_hot = pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=int)
    df = pd.concat((df.drop(col, axis=1), one_hot), axis=1)

In [ ]:
df.head()

In [129]:
for col in ['Category', 'Country']:
    mean_target = df.groupby(col)['target'].mean()
    df[col] = df[col].map(mean_target)

### Выделение признаков из текста

#### Лемматизация

In [71]:
data = df.copy()

In [ ]:
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')

In [ ]:
wnl = WordNetLemmatizer()
list1 = ['kites', 'babies', 'dogs', 'flys', 'flew', 'smiling',
         'driving', 'died', 'tried', 'feet']
for words in list1:
    print(words + " ---> " + wnl.lemmatize(words))

In [ ]:
data['Title']

In [86]:
clear_bag = []
for word in data['Title']:
    if type(word) == float:
        continue
    clear_bag.append(wnl.lemmatize(str(word.split())))

In [ ]:
clear_bag[0:10]

#### TF-IDF

In [89]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer()
tfidf.fit(clear_bag)
tfidf.vocabulary_[10]

In [ ]:
tfidf_ = pd.DataFrame(tfidf.transform([data['Title'][0]]).T.todense(),
                      index=tfidf.get_feature_names_out(),
                      columns=['tfidf'])

tfidf_.sort_values('tfidf', ascending=False)[:10]

In [ ]:
def tfidf_s(s):
    list_m = tfidf.transform([s]).T.todense()
    return list_m[list_m.nonzero()[0]].mean()


tfidf_s(data['Title'][0])

In [95]:
data['tf'] = data['Title'][0:1000].apply(tfidf_s)

In [ ]:
data.head()

In [ ]:
data[['target', 'tf']][0:1000].corr()

#### Sentiment Analysis

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('punkt')
nltk.download('vader_lexicon')

In [101]:
sid = SentimentIntensityAnalyzer()

In [ ]:
sentence = 'What is it to be a good doctor in a good hospital'
sid.polarity_scores(sentence)

In [106]:
def sent_a(s):
    ss = sid.polarity_scores(s)
    return ss['compound']

In [107]:
data['sentiment'] = data['Title'][0:1000].apply(sent_a)

In [ ]:
data.head(100)

In [ ]:
data[['target', 'tf', 'sentiment']][0:1000].corr()

In [ ]:
df.head()

### Разбиение датасета и импортирование библиотек

In [16]:
df = df.drop('Title', axis=1)

In [ ]:
df.head()

In [18]:
X = df.drop('target', axis=1)
Y = df['target']

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [20]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25)

In [ ]:
Y.value_counts()

### Логистическая регрессия

In [ ]:
pipe_log = Pipeline([('Scaler', StandardScaler()),
                    ('LR', LogisticRegression(class_weight='balanced'))])
pipe_log.fit(X_train, Y_train)

In [ ]:
pipe_log.classes_

In [ ]:
print(
    f'Accuracy равно: {accuracy_score(Y_test, pipe_log.predict(X_test)):.3f}')
print(
    f'Precision равно: {precision_score(Y_test, pipe_log.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe_log.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe_log.predict(X_test)):.3f}')

In [ ]:
pipe_mm = Pipeline([('Scaler', MinMaxScaler()), ('LR',
                   LogisticRegression(class_weight='balanced', solver='sag'))])
pipe_mm.fit(X_train, Y_train)

In [ ]:
print(f'Accuracy равно: {accuracy_score(Y_test, pipe_mm.predict(X_test)):.3f}')
print(
    f'Precision равно: {precision_score(Y_test, pipe_mm.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe_mm.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe_mm.predict(X_test)):.3f}')

### PR, AUC

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import auc
from sklearn.metrics import PrecisionRecallDisplay

precision_l, recall_l, thresholds_l = precision_recall_curve(
    Y_test, pipe_log.predict_proba(X_test)[:, 1])
precision_m, recall_m, thresholds_m = precision_recall_curve(
    Y_test, pipe_mm.predict_proba(X_test)[:, 1])

PrecisionRecallDisplay(precision=precision_l, recall=recall_l).plot()
PrecisionRecallDisplay(precision=precision_m, recall=recall_m).plot()

In [ ]:
# Посчитаем PR-AUC
auc(recall_l, precision_l)

In [ ]:
auc(recall_m, precision_m)

### Линейный SVM

In [ ]:
from sklearn.svm import LinearSVC
pipe = Pipeline([('Scaler', StandardScaler()),
                ('LR', LinearSVC(class_weight='balanced'))])
pipe.fit(X_train, Y_train)

In [ ]:
print(f'Accuracy равно: {accuracy_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Precision равно: {precision_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe.predict(X_test)):.3f}')

#### Подбор гиперпараметров модели

In [ ]:
pipe.get_params()

In [146]:
from sklearn.model_selection import GridSearchCV
params = {
    'LR__C': [1.0, 1.1, 1.2, 1.3],
    'LR__loss': ['hinge', 'squared_hinge'],
    'LR__penalty': ['l1', 'l2']
}

In [ ]:
search = GridSearchCV(pipe, params, cv=2)
search.fit(X_train, Y_train)
print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)

### Модель с подобранными параметрами

In [ ]:
pipe = Pipeline([('Scaler', StandardScaler()), ('LR', LinearSVC(
    class_weight='balanced', C=1.3, loss='hinge'))])
pipe.fit(X_train, Y_train)
print(f'Precision равно: {precision_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe.predict(X_test)):.3f}')

In [ ]:
pipe.predict_proba(X_test)

### CalibratedClassifier

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
clf = CalibratedClassifierCV(pipe)
clf.fit(X_train, Y_train)
print(f'Precision равно: {precision_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe.predict(X_test)):.3f}')

In [ ]:
clf.predict_proba(X_test.head(3))

In [ ]:
Y_test.head(3)

### Нелинейный SVM

In [ ]:
from sklearn.svm import SVC
pipe = Pipeline([('Scaler', StandardScaler()),
                ('LR', SVC(class_weight='balanced', probability=True))])
pipe.fit(X_train, Y_train)
print(f'Accuracy равно: {accuracy_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Precision равно: {precision_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe.predict(X_test)):.3f}')

### KNN - K-Nearest Neighbours

In [153]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
pipe = Pipeline([('Scaler', StandardScaler()),
                ('KNN', KNeighborsClassifier(n_neighbors=5))])
pipe.fit(X_train, Y_train)
print(f'Precision равно: {precision_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'Recall равно: {recall_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'F-мера равно: {f1_score(Y_test, pipe.predict(X_test)):.3f}')